# Colab Full Control MCP Setup

This notebook starts the MCP server inside Colab, checks it locally, opens a Cloudflare Tunnel, and prints the Codex MCP config snippet.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/CopyyQ/colab-full-control-mcp.git'
PROJECT_DIR = '/content/colab-full-control-mcp'
SRC_DIR = str(Path(PROJECT_DIR) / 'src')

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
%cd /content/colab-full-control-mcp


In [ ]:
%pip install -r requirements.txt
%pip install -e .

In [ ]:
import os
import sys
from pathlib import Path
from getpass import getpass

src_dir = str(Path('/content/colab-full-control-mcp/src'))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
existing_pythonpath = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = src_dir if not existing_pythonpath else src_dir + ':' + existing_pythonpath

os.environ['COLAB_MCP_TOKEN'] = getpass('COLAB_MCP_TOKEN: ')
os.environ['PERMISSION_PROFILE'] = 'DEVELOPER'
os.environ['ALLOWED_ROOTS'] = '/content,/content/drive/MyDrive'
os.environ['UNRESTRICTED_RUNTIME_MODE'] = 'false'
print('Kernel src path ready:', src_dir)


In [ ]:
import subprocess, sys

server_proc = subprocess.Popen(
    [sys.executable, 'scripts/start_server.py'],
    cwd='/content/colab-full-control-mcp',
)
print('server pid =', server_proc.pid)

In [ ]:
!python scripts/health_check.py

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb || apt-get -f install -y
!python scripts/start_tunnel.py --server-url http://127.0.0.1:8000

In [ ]:
import json
from pathlib import Path

state = json.loads(Path('/content/.colab_full_control_mcp/jobs/cloudflared_state.json').read_text())
public_url = state['url'] + '/mcp'
print('Public MCP URL:', public_url)
!python scripts/print_codex_config.py --url {public_url}

In [ ]:
import sys
from pathlib import Path

src_dir = str(Path('/content/colab-full-control-mcp/src'))
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from colab_full_control_mcp.tools import TOOL_REGISTRY

print('Registered tools:', sum(len(names) for names in TOOL_REGISTRY.values()))
print('Import path:', src_dir)


In [ ]:
!python scripts/stop_tunnel.py
!python scripts/stop_server.py